In [1]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import pandas as pd
import math
import random
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")

class HEMSEnvWithNetMetering(gym.Env):
    """
    The definitive HEMS environment with net metering, optimized by a GA-RL hybrid.
    """
    metadata = {"render.modes": ["human"]}

    def __init__(self, config):
        super().__init__()
        # Load data
        self.solar_df = pd.read_csv(config["solar_path"], parse_dates=["time"])
        self.load_df = pd.read_csv(config["load_path"], parse_dates=["Date_Time"])
        self.solar_df.columns = self.solar_df.columns.str.strip()
        self.load_df.columns = self.load_df.columns.str.strip()

        # Config and hardware
        self.config = config
        self.solar_panel_area = config["solar_panel_area"]
        self.max_battery = float(config["max_battery_kwh"])
        self.appliances = config["appliances"]
        self.feed_in_tariff = config.get("feed_in_tariff", 12.0) # New parameter for selling price
        
        # Reward weights from GA
        self.w_grid_cost = config["reward_weights"]["grid"]
        self.w_battery_deg = config["reward_weights"]["battery_deg"]
        self.w_comfort = config["reward_weights"]["comfort"]
        
        self.n_steps = min(len(self.load_df), len(self.solar_df))
        self.dt_hours = 1 / 60.0
        self.steps_per_day = 24 * 60

        # Gym spaces
        self.action_space = spaces.Discrete(2**len(self.appliances))
        num_obs_features = 4 + len(self.appliances)
        self.observation_space = spaces.Box(low=-1.0, high=1.0, shape=(num_obs_features,), dtype=np.float32)
        
        self.time_step = 0
        self.battery_soc = 0.0
        self._reset_appliance_states()
        self.episode_info = []

    def _reset_appliance_states(self):
        self.appliance_states = []
        for app in self.appliances:
            self.appliance_states.append({"name": app["name"], "is_running": False, "steps_remaining": 0, "task_completed_today": False})

    def _compute_solar_output_kw(self, solar_row):
        ghi = pd.to_numeric(solar_row.get("ghi_pyr"), errors='coerce')
        humidity = pd.to_numeric(solar_row.get("relative_humidity"), errors='coerce')
        if pd.isna(ghi) or pd.isna(humidity): return 0.0
        panel_efficiency = 0.18
        weather_factor = max(0.0, 1.0 - (humidity / 100.0))
        pv_kw = (ghi * self.solar_panel_area * panel_efficiency * weather_factor) / 1000.0
        return max(pv_kw, 0.0)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.time_step = 0
        self.battery_soc = 0.5 * self.max_battery
        self._reset_appliance_states()
        self.episode_info = []
        obs, info = self._get_obs()
        return obs, info

    def _get_obs(self):
        idx = self.time_step % self.n_steps
        solar_row = self.solar_df.iloc[idx]
        solar_kw = self._compute_solar_output_kw(solar_row)
        soc_norm = self.battery_soc / self.max_battery if self.max_battery > 0 else 0.0
        hour = (self.time_step // 60) % 24
        hour_rad = (2.0 * math.pi * hour) / 24.0
        appliance_statuses = [1.0 if not s["task_completed_today"] else 0.0 for s in self.appliance_states]
        obs_list = [solar_kw, soc_norm, math.sin(hour_rad), math.cos(hour_rad)] + appliance_statuses
        return np.array(obs_list, dtype=np.float32), {}

    def _get_grid_price_for_hour(self, hour):
        return 46.85 if 19 <= hour <= 23 else 40.53

    def step(self, action):
        idx = self.time_step % self.n_steps
        current_load_row = self.load_df.iloc[idx]
        
        if self.time_step > 0 and self.time_step % self.steps_per_day == 0:
            for state in self.appliance_states:
                state["task_completed_today"] = False

        action_commands = [int(bit) for bit in np.binary_repr(action, width=len(self.appliances))]
        appliance_load_kw = 0
        for i, start_command in enumerate(action_commands):
            state = self.appliance_states[i]
            if start_command == 1 and not state["is_running"] and not state["task_completed_today"]:
                state["is_running"] = True
                state["steps_remaining"] = self.appliances[i]["duration_steps"]
                state["task_completed_today"] = True
            if state["is_running"]:
                appliance_load_kw += self.appliances[i]["power_kw"]
                state["steps_remaining"] -= 1
                if state["steps_remaining"] == 0: state["is_running"] = False

        base_load_kw = current_load_row['Refrigerator_kW'] + current_load_row['WD_kW'] + current_load_row['Kitchen_light1_kW'] + current_load_row['Kitchen_light2_kW'] + current_load_row['SR_kW']
        total_load_kwh = (base_load_kw + appliance_load_kw) * self.dt_hours
        
        solar_row = self.solar_df.iloc[idx]
        solar_kwh_available = self._compute_solar_output_kw(solar_row) * self.dt_hours
        
        grid_energy_import, battery_discharge, grid_export_kwh = 0.0, 0.0, 0.0
        remaining_load = total_load_kwh
        solar_to_load = min(solar_kwh_available, remaining_load)
        remaining_load -= solar_to_load
        
        if self.battery_soc > 0 and remaining_load > 0:
            discharge = min(self.battery_soc, remaining_load / 0.95)
            battery_discharge = discharge
            self.battery_soc -= discharge
            remaining_load -= discharge * 0.95
        
        if remaining_load > 0:
            grid_energy_import = remaining_load
        
        excess_solar = solar_kwh_available - solar_to_load
        if excess_solar > 0:
            charge = min(excess_solar * 0.95, self.max_battery - self.battery_soc)
            self.battery_soc += charge
            # *** NEW: Sell any remaining solar to the grid ***
            remaining_excess = excess_solar - (charge / 0.95)
            if remaining_excess > 0:
                grid_export_kwh = remaining_excess

        hour = (self.time_step // 60) % 24
        grid_cost = grid_energy_import * self._get_grid_price_for_hour(hour)
        grid_revenue = grid_export_kwh * self.feed_in_tariff
        battery_deg_cost = battery_discharge * self.config.get("battery_deg_cost_per_kwh", 0.01)
        
        comfort_penalty = 0
        missed_task_today = False
        if (self.time_step + 1) % self.steps_per_day == 0:
            if not all(s["task_completed_today"] for s in self.appliance_states):
                comfort_penalty = 1
                missed_task_today = True
        
        # The reward now includes revenue from selling to the grid
        reward = -((self.w_grid_cost * grid_cost) + (self.w_battery_deg * battery_deg_cost) + (self.w_comfort * comfort_penalty)) + grid_revenue
        
        terminated = self.time_step >= (self.n_steps - 1)
        info = { "total_cost": grid_cost + battery_deg_cost, "grid_revenue": grid_revenue, "task_missed": missed_task_today }
        
        self.time_step += 1
        obs, _ = self._get_obs()
        return obs, reward, terminated, False, info

def analyze_and_define_jobs(data_path):
    df = pd.read_csv(data_path)
    df.columns = df.columns.str.strip()
    df.fillna(0, inplace=True)
    appliances = []
    
    schedulable_cols = ['Laundary_kW', 'AC_BR_kW', 'AC_GR_kW', 'AC_kW', 'AC_MBR_kW', 'WP_kW']
    for col in schedulable_cols:
        power_draw = df[df[col] > 0.1][col].mean()
        if pd.notna(power_draw) and power_draw > 0.1:
             appliances.append({"name": col, "power_kw": power_draw, "duration_steps": df[col].gt(0.1).sum() // df[col].gt(0.1).diff().ne(0).sum()})
    return appliances

def run_rl_for_fitness(config, training_timesteps=20000):
    try:
        def make_env(): return HEMSEnvWithNetMetering(config)
        vec_env = DummyVecEnv([make_env])
        vec_env = VecNormalize(vec_env, norm_obs=True, norm_reward=False, clip_obs=10.0)
        model = PPO("MlpPolicy", vec_env, verbose=0)
        model.learn(total_timesteps=training_timesteps)
        all_infos = []
        obs = vec_env.reset()
        done = False
        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, _, terminated, info = vec_env.step(action)
            all_infos.append(info[0])
            done = terminated[0]
        results = pd.DataFrame(all_infos)
        net_cost = results['total_cost'].sum() - results['grid_revenue'].sum()
        missed_tasks = results['task_missed'].sum()
        return net_cost + (missed_tasks * 1_000_000)
    except Exception: return float('inf')

def genetic_algorithm_optimizer(base_config, n_generations=10, pop_size=10):
    print("--- Starting GA Optimization of RL Agent with Net Metering ---")
    bounds = [[0.1, 10.0], [0.0, 5.0], [500, 20000]]
    population = [[random.uniform(b[0], b[1]) for b in bounds] for _ in range(pop_size)]
    best_weights, best_fitness = None, float('inf')
    for gen in range(n_generations):
        print(f"\n> GA Generation {gen+1}/{n_generations}")
        fitness_scores = [run_rl_for_fitness(base_config | {"reward_weights": {"grid": w[0], "battery_deg": w[1], "comfort": w[2]}}) for w in population]
        current_best_idx = np.argmin(fitness_scores)
        if fitness_scores[current_best_idx] < best_fitness:
            best_fitness, best_weights = fitness_scores[current_best_idx], population[current_best_idx]
        print(f"  - Best Net Cost Found: {best_fitness:,.2f} PKR (from weights {np.round(best_weights, 2)})")
        
        children = [population[np.argmin(fitness_scores)]]
        while len(children) < pop_size:
            tournament_indices = random.sample(range(pop_size), 3)
            p1 = population[tournament_indices[np.argmin([fitness_scores[i] for i in tournament_indices])]]
            tournament_indices = random.sample(range(pop_size), 3)
            p2 = population[tournament_indices[np.argmin([fitness_scores[i] for i in tournament_indices])]]
            crossover_point = random.randint(1, len(p1) - 1)
            child = p1[:crossover_point] + p2[crossover_point:]
            for j in range(len(child)):
                if random.random() < 0.2:
                    child[j] += random.uniform(-0.5, 0.5) * (bounds[j][1] - bounds[j][0])
                    child[j] = max(bounds[j][0], min(bounds[j][1], child[j]))
            children.append(child)
        population = children
    return best_weights, best_fitness

if __name__ == "__main__":
    appliance_jobs = analyze_and_define_jobs("homee.csv")
    base_config = {"load_path": "homee.csv", "solar_path": "solar.csv", "solar_panel_area": 27.8, "max_battery_kwh": 10.0, "appliances": appliance_jobs, "feed_in_tariff": 12.0}
    
    optimal_weights, final_net_cost = genetic_algorithm_optimizer(base_config, n_generations=5, pop_size=5)
    
    print("\n\n=============================================="); print("===      GA-RL Hybrid (Net Metering)       ==="); print("==============================================")
    print("\nGenetic Algorithm has found the optimal reward weights.")
    print(f"\nOptimal Reward Weights: Grid Cost: {optimal_weights[0]:.2f}, Battery Degradation: {optimal_weights[1]:.2f}, Comfort Penalty: {optimal_weights[2]:.2f}")
    print(f"\nRL agent trained with these weights achieved a final Net Annual Bill of: {final_net_cost:,.2f} PKR")

--- Starting GA Optimization of RL Agent with Net Metering ---

> GA Generation 1/5
  - Best Net Cost Found: 34,133.51 PKR (from weights [1.98000e+00 1.48000e+00 1.06496e+04])

> GA Generation 2/5
  - Best Net Cost Found: 34,103.07 PKR (from weights [  1.98   1.48 500.  ])

> GA Generation 3/5
  - Best Net Cost Found: 34,043.09 PKR (from weights [1.98000e+00 2.57000e+00 4.11077e+03])

> GA Generation 4/5
  - Best Net Cost Found: 34,043.09 PKR (from weights [1.98000e+00 2.57000e+00 4.11077e+03])

> GA Generation 5/5
  - Best Net Cost Found: 34,043.09 PKR (from weights [1.98000e+00 2.57000e+00 4.11077e+03])


===      GA-RL Hybrid (Net Metering)       ===

Genetic Algorithm has found the optimal reward weights.

Optimal Reward Weights: Grid Cost: 1.98, Battery Degradation: 2.57, Comfort Penalty: 4110.77

RL agent trained with these weights achieved a final Net Annual Bill of: 34,043.09 PKR
